# VCG Loss and PenaltyQAOA Tuning

This notebook reads saved results and exports figures. Error bars on mean plots show **95% confidence intervals for the mean** across constraints in Study A or COPs in Study B:

$$\bar y\;\pm\;t_{0.975,\,n-1}\frac{s}{\sqrt n},$$

where $s$ is the sample standard deviation and $n$ is the number of contributing instances. Optimizer restarts and measurement shots are not counted as independent instances. These are pointwise Student's t intervals, with no bootstrap or multiple-comparison adjustment. Intervals are omitted when fewer than two instances contribute.

The intervals approximate uncertainty in the mean for the represented problem mix. Small or skewed groups can have unreliable coverage; they do not establish that one method is better than another. Bounds are not clipped to $[0,1]$. Scatter plots, individual training histories, and dataset or state distributions show individual observations rather than means and therefore have no mean confidence intervals. Tables retain explicitly named standard-deviation columns as descriptive statistics.

Smoke results use short training runs to check the code.

In [ ]:
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "code-tuning"]
REPO = next(p for p in candidates if (p / "tuning/config.json").exists())
os.environ.setdefault("MPLCONFIGDIR", str(REPO / "tuning/.cache/matplotlib"))
sys.path.insert(0, str(REPO / "tuning"))
from analyze import read_results, status_summary, ranking, FigureWriter, figures
from IPython.display import display, Image

# Edit this path to compare a different saved run.
RESULTS_DIR = Path(os.environ.get("PCQAOA_TUNING_RESULTS", str(REPO / "tuning/results/full")))
if not (RESULTS_DIR / "run.json").exists():
    RESULTS_DIR = REPO / "tuning/results/smoke"
# Study B has its own directory so the existing Study A run stays intact.
B_RESULTS_DIR = Path(os.environ.get("PCQAOA_TUNING_B_RESULTS",
    str(REPO / "tuning/results/full-b-five-methods" if RESULTS_DIR.name == "full" else RESULTS_DIR)))
if not (B_RESULTS_DIR / "run.json").exists():
    B_RESULTS_DIR = RESULTS_DIR
data = read_results(RESULTS_DIR, B_RESULTS_DIR)
writers = {"a": FigureWriter(RESULTS_DIR / "figures", latex=True),
           "b": FigureWriter(B_RESULTS_DIR / "figures", latex=True)}

def show_section(section, study):
    writer = writers[study]
    for path in figures(data, section, writer, study=study):
        display(Image(filename=str(path), width=900))
    print("Fonts:", writer.font_status)


## Study A: VCG Loss Selection

### Progress and Dataset Composition

Rerun the next cell to refresh this study. A completed VCG task can have fidelity below 0.999.

In [ ]:
data = read_results(RESULTS_DIR, B_RESULTS_DIR)
print("Results:", RESULTS_DIR)
print(json.dumps(status_summary(RESULTS_DIR)["a"], indent=2))
instances = data["a_instances"]
if not instances.empty:
    display(instances[['id', 'support', 'family', 'feasible_count', 'feasible_fraction']])
show_section("datasets", "a")

### Loss Functions and Fidelity

Let $\mathcal F$ be the feasible set and $|\psi_\theta\rangle$ the trained state. The target is

$$|u_{\mathcal F}\rangle=\frac{1}{\sqrt{|\mathcal F|}}\sum_{x\in\mathcal F}|x\rangle.$$

The three state measurements are

$$P_{\mathcal F}=\sum_{x\in\mathcal F}|\langle x|\psi_\theta\rangle|^2,\qquad
F=|\langle u_{\mathcal F}|\psi_\theta\rangle|^2,\qquad
F_{\mathrm{cond}}=\frac{F}{P_{\mathcal F}}.$$

The code sets $F_{\mathrm{cond}}=0$ when $P_{\mathcal F}\leq10^{-12}$ to avoid division by zero.

The run compares **five loss conditions**, using three formulas:

$$\begin{aligned}
L_{\mathrm{feas}} &= 1-2P_{\mathcal F},\\
L_{\mathrm{fid}} &= 1-F,\\
L_{0.5} &= (1-P_{\mathcal F})+0.5(1-F_{\mathrm{cond}}),\\
L_1 &= (1-P_{\mathcal F})+(1-F_{\mathrm{cond}}),\\
L_2 &= (1-P_{\mathcal F})+2(1-F_{\mathrm{cond}}).
\end{aligned}$$

At each depth, retain the restart with the lowest final training loss. Stop when its fidelity reaches 0.999 or after depth 8. Return the highest-fidelity state among the retained depths. Select the loss with the highest mean returned fidelity across all 50 constraints.

Depth plots include only tasks that reached each depth. Early stopping can change which constraints contribute to those means. Rankings shown before all tasks finish are incomplete.

In [ ]:
if not data["a"].empty:
    display(ranking(data["a"], "fidelity"))
    display(data["a"][["instance_id", "condition", "fidelity", "p_feasible", "conditional_fidelity",
                       "entropy", "coverage", "min_conditional_probability", "selected_depth", "converged"]])
show_section("loss", "a")

### Example States and Optimization Histories

Each distribution title shows the constraint and training loss. Blue bars represent feasible states; rose bars represent infeasible states. The examples have low, middle and high returned fidelity.

Training histories record loss before each indicated update. Saved final metrics are calculated after the final update.

The training-history plot shows one constraint at one ma-QAOA depth, identified in the title along with the loss function. Each curve is a separate optimizer restart. Color and line style identify the restart in the legend. The thicker curve marked **Retained** is the restart with the lowest final training loss at that depth. It need not be the restart with the highest fidelity. The horizontal axis counts Adam updates; the vertical axis is the training loss, not fidelity. Only saved history points are plotted.


In [ ]:
show_section("examples", "a")

### Constraint Resource Estimates

We count the gates needed to prepare each saved VCG. The counts use its selected depth and trained angles. No VCG is retrained.

- **NISQ** means noisy intermediate-scale quantum. Here, gates are counted using $\{R_X,R_Y,R_Z,H,\mathrm{CNOT},X,Y,Z,T,S\}$.
- **FTQC** means fault-tolerant quantum computing. Here, rotations are approximated using $\{H,T,\mathrm{CNOT},S\}$. These counts exclude error correction and physical hardware costs.
- The first two figures compare CNOT counts and T counts across the five losses.
- The next two compare **one selected VCG preparation** with **one penalty cost layer**. Repeated penalty terms are combined before counting. These are not full PC-QAOA versus PenaltyQAOA comparisons: Grover mixers also undo and repeat gadget preparation in every layer.
- `n_allocated` counts decision and slack qubits. Extra qubits used by the estimator are listed separately.

Individual gate counts are integers. Tables and curves show averages, which can have decimals. Error bars are 95% confidence intervals across constraints.

Counts use PennyLane's gate decompositions, without removing zero-angle gates. FTQC rotation precision is recorded in the saved estimates. Results are cached in `resource_estimates/` and recalculated when their inputs or estimator settings change.

In [ ]:
from resource_analysis import estimate_study_a, estimate_study_b, resource_figures
resource_a = estimate_study_a(RESULTS_DIR)
selection_path = RESULTS_DIR / "selected_loss.json"
selected_loss = json.loads(selection_path.read_text())["condition"] if selection_path.exists() else "fidelity"
print("Resource Rows:", len(resource_a))
if not resource_a.empty:
    display(resource_a.groupby(["representation", "status", "gate_set"]).size().rename("Rows"))
    display(resource_a.groupby(["gate_set", "condition", "representation"])[
        ["total_gates", "two_qubit_gates", "t_gates", "n_allocated", "total_wires"]].mean())
    display(resource_a[["instance_id", "condition", "representation", "gate_set",
        "selected_depth", "total_gates", "two_qubit_gates", "t_gates", "n_slack",
        "n_allocated", "algo_wires", "zeroed_wires", "any_state_wires"]])
    for path in resource_figures(resource_a, "a", writers["a"], selected_loss):
        display(Image(filename=str(path), width=900))
    print("Fonts:", writers["a"].font_status)


### Runtime

Compilation and optimizer-update times are recorded separately. Peak RSS is the largest resident memory use of the worker process.

In [ ]:
frame = data["a"]
if not frame.empty:
    display(frame.groupby("condition")[["compile_seconds", "optimize_seconds", "process_peak_rss_mb"]].agg(["mean", "std"]))

### Saved Selection and Reproducibility

The selection file is written after all tasks in this study finish.

In [ ]:
path = RESULTS_DIR / "selected_loss.json"
print(path.read_text() if path.exists() else "No final selection yet.")
print("Environment:")
print(json.dumps(data["run"]["environment"], indent=2))
print("Figure Directory:", writers["a"].directory)

## Study B: Penalty Selection

### Progress and Dataset Composition

Rerun the next cell to refresh this study. Feasible fractions count decision strings satisfying all COP constraints.

In [ ]:
data = read_results(RESULTS_DIR, B_RESULTS_DIR)
print("Results:", B_RESULTS_DIR)
print(json.dumps(status_summary(B_RESULTS_DIR)["b"], indent=2))
instances = data["b_instances"]
if not instances.empty:
    display(instances[['id', 'n_x', 'regime', 'families', 'n_structural', 'n_penalty', 'n_slack', 'feasible_fraction']])
show_section("datasets", "b")

### Penalty Methods and Optimal-Solution Probability

Write the objective with each coefficient counted once:

$$f(x)=\sum_i q_i x_i+\sum_{i<j}q_{ij}x_ix_j.$$

Here $q_i=Q_{ii}$ and $q_{ij}=Q_{ij}$ for the saved upper-triangular matrix. Set $c_{ij}=q_{\min(i,j),\max(i,j)}$ for $i\ne j$. Let $q_t$ range over all objective coefficients and define

$$L_Q=\sum_t\min(0,q_t),\qquad U_Q=\sum_t\max(0,q_t).$$

Study B compares **five fixed penalty rules**. There are no multipliers and no manuscript-reference setting:

$$\begin{aligned}
\delta_{\mathrm{range}} &= U_Q-L_Q+1,\\
\delta_{\mathrm{feasible}} &= f(\bar x)-L_Q+1,\\
\delta_{\mathrm{VL}} &= 1+\max_i\left\{q_i+\sum_{j\ne i}\max(c_{ij},0),\;-q_i-\sum_{j\ne i}\min(c_{ij},0)\right\},\\
\delta_{\mathrm{local}} &= 1+\max_i\left(|q_i|+\sum_{j\ne i}|c_{ij}|\right),\\
\delta_{\mathrm{max}} &= 1+\max_t|q_t|.
\end{aligned}$$

The feasible point $\bar x$ is the first feasible string found in a seeded random ordering of decision strings, searched without replacement. The search checks constraints only and stops immediately on success. Its seed, number of checks and elapsed time are saved. The objective is evaluated only after the point is chosen. This enumeration-based search is intended for the small calibration COPs; finding a feasible point can be expensive on larger problems.

The range and feasible-solution bounds are sufficient for the integer squared-residual encoding. Verma–Lewis, absolute local change and maximum coefficient are heuristics for these general overlapping constraints. The signed local calculation follows [Verma and Lewis (2022)](https://doi.org/10.1016/j.disopt.2020.100594); the sufficient bounds use the objective-bound reasoning discussed by [Diez García et al. (2022)](https://doi.org/10.1145/3520304.3528925).

None of these rules uses an optimum. Exact optimal solutions are calculated separately to measure performance. For the set $\mathcal X^\star$ of all optimal feasible decision strings,

$$P(\mathrm{opt})=\sum_{x\in\mathcal X^\star}P(x).$$

Slack registers are marginalized before calculating this probability. Standard QAOA runs at depths 1–5, with 10 restarts and 50 Adam steps per restart at learning rate 0.01. At each depth, retain the restart with the lowest final penalized expectation. Select the method with the highest mean sampled $P(\mathrm{opt})$ at depth 5, using 10,000 shots per COP. All five methods are eligible: **20 COPs × five methods = 100 tasks**. Exact probabilities are shown for comparison. Smoke runs use shorter budgets.

In [ ]:
if not data["b"].empty:
    display(ranking(data["b"], "p_optimal"))
    display(data["b"][["instance_id", "condition", "delta", "selectable", "depth", "p_optimal",
                       "p_feasible", "exact_p_optimal", "exact_p_feasible"]])
show_section("penalty", "b")
if not data["b_instances"].empty:
    search_columns = ["id", "feasible_point", "feasibility_search_seed",
                      "feasibility_search_attempts", "feasibility_search_seconds"]
    display(data["b_instances"].reindex(columns=search_columns))

### Constraint and Circuit Resource Estimates

The tables count each constraint as a gadget and as one penalty cost layer. A constraint without an exact gadget or saved VCG is marked `missing_vcg`. It is not retrained or assigned a zero gate count.

- Full PenaltyQAOA counts include initial Hadamards on all decision and slack qubits, objective and penalty gates, and mixers.
- Repeated Pauli terms are combined as they were during Study B training.
- At depth $p$, the total is $G_{\mathrm{prep}}+pG_{\mathrm{layer}}$: initial preparation plus $p$ cost-and-mixer layers.
- The five penalty rules change rotation angles but use the same gates, so their resource counts match. Figures count each COP once.
- The NISQ and FTQC figures show depth-five costs. Points identify disjoint and overlapping COPs; the mean and 95% interval use four COPs per decision size.
- The merging figure compares layer counts before and after combining repeated terms. Points below the diagonal show fewer gates after merging.

NISQ and FTQC use the gate sets listed in Study A. Individual counts are integers; averages can have decimals. These figures do not provide a full PC-QAOA comparison.

Change `VCG_DB_PATH` below to use another gadget database. Tables and cached estimates are saved in Study B's `resource_estimates/` directory.

In [ ]:
VCG_DB_PATH = RESULTS_DIR / "selected_vcg_db.pkl"
if not VCG_DB_PATH.exists():
    VCG_DB_PATH = REPO / "tuning/artifacts/selected_vcg_db.pkl"
resource_b = estimate_study_b(B_RESULTS_DIR, VCG_DB_PATH)
print("VCG Database:", VCG_DB_PATH)
print("Resource Rows:", len(resource_b))
if not resource_b.empty:
    display(resource_b.groupby(["representation", "status", "gate_set"]).size().rename("Rows"))
    missing = resource_b[resource_b.status == "missing_vcg"]
    if not missing.empty:
        print("Constraint Occurrences Without a Saved VCG:",
              len(missing[["instance_id", "constraint_index"]].drop_duplicates()))
        display(missing[["instance_id", "constraint", "family"]].drop_duplicates())
    circuits = resource_b[resource_b.representation == "full_penalty_qaoa"]
    display(circuits.groupby(["gate_set", "condition", "depth"])[
        ["total_gates", "two_qubit_gates", "t_gates", "n_allocated"]].mean())
    display(circuits[["instance_id", "condition", "gate_set", "depth", "delta",
        "total_gates", "prep_gates", "layer_gates", "expanded_layer_gates",
        "n_allocated", "n_slack", "algo_wires", "zeroed_wires", "any_state_wires"]])
    for path in resource_figures(resource_b, "b", writers["b"]):
        display(Image(filename=str(path), width=900))
    print("Fonts:", writers["b"].font_status)


### Qubits and Runtime

PC-QAOA slack counts come from the constraint partition; Study B runs only PenaltyQAOA.

Compilation and optimizer-update times are recorded separately. Peak RSS is the largest resident memory use of the worker process.

In [ ]:
show_section("resources", "b")
frame = data["b"]
if not frame.empty:
    display(frame.groupby("condition")[["compile_seconds", "optimize_seconds", "process_peak_rss_mb"]].agg(["mean", "std"]))

### Saved Selection and Reproducibility

The selection file is written after all tasks in this study finish.

In [ ]:
path = B_RESULTS_DIR / "selected_penalty.json"
print(path.read_text() if path.exists() else "No final selection yet.")
print("Environment:")
print(json.dumps(data.get("b_run", data["run"])["environment"], indent=2))
print("Figure Directory:", writers["b"].directory)

## Figure Exports

Figures are saved in each study’s own `figures/` directory as PDF and 300-dpi PNG files. Rendering uses LaTeX Computer Modern when available, with Matplotlib Computer Modern math and DejaVu Serif as the fallback. Each directory records its font choice in `font_status.json`.